<a href="https://colab.research.google.com/github/abhsrivastava/hugging_face_transformers/blob/main/Full_End_to_End_Classification_Example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Enhanced Hugging Face Sequence Classification

# This notebook builds a practical customer-feedback sentiment-classification workflow using Hugging Face Transformers.

# It demonstrates:

# - Loading a task-specific sequence-classification model
# - Inspecting model configuration and label mappings
# - Understanding why chat templates are not used
# - Tokenizing multiple inputs as a batch
# - Dynamic padding and attention masks
# - Running inference with `torch.no_grad()`
# - Understanding logits, softmax, and argmax
# - Displaying results in a Pandas DataFrame
# - Handling low-confidence predictions
# - Applying example business-routing rules
# - Testing multilingual and difficult inputs
# - Comparing manual inference with `pipeline()`
# - Comparing batch inference with one-at-a-time inference
# - Demonstrating truncation

# The model used is:

# `tabularisai/multilingual-sentiment-analysis`

# It predicts five sentiment classes:

# 1. Very Negative
# 2. Negative
# 3. Neutral
# 4. Positive
# 5. Very Positive

# > The model is licensed under CC-BY-NC-4.0. Review its license before using it commercially.

In [2]:
#1. Load the dependencies

%pip install -q --upgrade \
    transformers \
    datasets \
    evaluate \
    accelerate \
    scikit-learn \
    pandas

print(f'✅ Installed Dependencies Successfully')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 9.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
✅ Installed Dependencies Successfully


In [4]:
# Imports

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    pipeline
)
import pandas as pd
import torch
import time

print(f'✅ Imports done successfully')

print(f'Pytorch version: {torch.__version__}')
print(f'Cuda Availablele: {torch.cuda.is_available()}')

✅ Imports done successfully
Pytorch version: 2.11.0+cpu
Cuda Availablele: False


In [5]:
# Now let us load the tokenizer and the model
# For classification, we need a model with a trained classification head. Therefore, this notebook loads the checkpoint with:
# `AutoModelForSequenceClassification`

CHECKPOINT = "tabularisai/multilingual-sentiment-analysis"
print(f"Checkpoint: {CHECKPOINT}")

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT)

print(f'Tokenizer: {tokenizer.__class__.__name__}')
print(f'Model: {model.__class__.__name__}')

Checkpoint: tabularisai/multilingual-sentiment-analysis


config.json:   0%|          | 0.00/851 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.92M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  541MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Tokenizer: DistilBertTokenizer
Model: DistilBertForSequenceClassification


In [7]:
#Select a GPU if available
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Move the Model to Selected Device
model = model.to(device)

# Put the model in eval mode so that training specific behavior such as dropout is disabled.
model.eval()

print(f"Running on: {device}")

Running on: cpu


In [34]:
## 3. Inspect the model configuration

# The configuration tells us:

# - Which architecture was loaded
# - How many output labels exist
# - How numeric output positions map to sentiment labels
# - The model's maximum sequence length

config = model.config

# Record model information
# A production project should record the model ID, task, label set, license, and important limitations

model_information = {
    "checkpoint": CHECKPOINT,
    "task": "sentiment classification",
    "architecture": config.architectures[0],
    "number_of_labels": config.num_labels,
    "labels": list(config.id2label.values()),
    "license": "CC-BY-NC-4.0",
    "commercial_use_requires_review": True,
    "requires_chat_template": False,
    "maximum_positions": config.max_position_embeddings
}

for key, value in model_information.items():
    print(f"{key}: {value}")

print("\nLabel mappings")
print("=" * 60)

for label_id, label_name in config.id2label.items():
    print(f"Output position {label_id}: {label_name}")

checkpoint: tabularisai/multilingual-sentiment-analysis
task: sentiment classification
architecture: DistilBertForSequenceClassification
number_of_labels: 5
labels: ['Very Negative', 'Negative', 'Neutral', 'Positive', 'Very Positive']
license: CC-BY-NC-4.0
commercial_use_requires_review: True
requires_chat_template: False
maximum_positions: 512

Label mappings
Output position 0: Very Negative
Output position 1: Negative
Output position 2: Neutral
Output position 3: Positive
Output position 4: Very Positive


In [9]:
# Check if the model Needs a Chat template

chat_template = getattr(tokenizer, "chat_template", None)

if chat_template is None:
    print("✅ No chat template is expected.")
    print("Pass ordinary text directly to this classifier.")
else:
    print("This tokenizer contains a chat template:")
    print(chat_template)

✅ No chat template is expected.
Pass ordinary text directly to this classifier.


In [23]:
# Create Input Data
customer_messages = [
    "The new version is fantastic. Everything works perfectly!",
    "My payment failed twice and support has not responded for three days.",
    "The transfer completed successfully.",
    "The app is okay, but the new interface takes some time to understand.",
    "This is the worst banking experience I have ever had. ",
    "The app constantly crashes and I cannot access my money.",
    "The customer service was excellent!",
    "El servicio al cliente fue excelente!",
    "Le service client était excellent !",
    "Der Kundenservice war ausgezeichnet!",
    "The payment experience was terrible.",
    "El proceso de pago fue terrible.",
    "Le processus de paiement était terrible."
    "The application keeps crashing whenever I attempt to make a payment. " * 20,
]

print(f"Number of messages: {len(customer_messages)}")

for message_number, message in enumerate(customer_messages, start=1):
    print(f"\n{message_number}. {message}")

Number of messages: 13

1. The new version is fantastic. Everything works perfectly!

2. My payment failed twice and support has not responded for three days.

3. The transfer completed successfully.

4. The app is okay, but the new interface takes some time to understand.

5. This is the worst banking experience I have ever had. 

6. The app constantly crashes and I cannot access my money.

7. The customer service was excellent!

8. El servicio al cliente fue excelente!

9. Le service client était excellent !

10. Der Kundenservice war ausgezeichnet!

11. The payment experience was terrible.

12. El proceso de pago fue terrible.

13. Le processus de paiement était terrible.The application keeps crashing whenever I attempt to make a payment. Le processus de paiement était terrible.The application keeps crashing whenever I attempt to make a payment. Le processus de paiement était terrible.The application keeps crashing whenever I attempt to make a payment. Le processus de paiement était

In [24]:
## 6. Batch tokenization and Attention Masking

# `padding="longest"` pads every input to the length of the longest input in the current batch.

# `truncation=True` removes tokens beyond `max_length`.

# `return_tensors="pt"` returns PyTorch tensors.


encoded_batch = tokenizer(
    customer_messages,
    return_tensors="pt",
    padding="longest",
    truncation=True,
    max_length=512
)

input_ids = encoded_batch["input_ids"]
attention_mask = encoded_batch["attention_mask"]

print("Input IDs shape:", input_ids.shape)
print("Attention mask shape:", attention_mask.shape)

Input IDs shape: torch.Size([13, 442])
Attention mask shape: torch.Size([13, 442])


In [25]:
# Inspect the tokens and mask for the shortest message

token_counts = attention_mask.sum(dim=1)

shortest_message_index = torch.argmin(token_counts).item()

shortest_input_ids = input_ids[shortest_message_index]
shortest_attention_mask = attention_mask[shortest_message_index]

tokens = tokenizer.convert_ids_to_tokens(
    shortest_input_ids
)

token_inspection = []

for position, (token, token_id, mask_value) in enumerate(
    zip(
        tokens,
        shortest_input_ids.tolist(),
        shortest_attention_mask.tolist()
    )
):
    token_inspection.append({
        "position": position,
        "token": token,
        "token_id": token_id,
        "attention_mask": mask_value,
        "position_type": (
            "real token"
            if mask_value == 1
            else "padding"
        )
    })

print("Shortest message:")
print(customer_messages[shortest_message_index])

pd.DataFrame(token_inspection)

Shortest message:
The transfer completed successfully.


,position,token,token_id,attention_mask,position_type
0,0,[CLS],101,1,real token
1,1,The,10117,1,real token
2,2,transfer,21110,1,real token
3,3,completed,15782,1,real token
4,4,successfully,32094,1,real token
...,...,...,...,...,...
437,437,[PAD],0,0,padding
438,438,[PAD],0,0,padding
439,439,[PAD],0,0,padding
440,440,[PAD],0,0,padding


In [26]:
# Move the tensors to the model's device
# The tokenizer initially creates CPU tensors. If the model is using a GPU, the inputs must also be moved to that GPU.

model_inputs = {
    tensor_name: tensor.to(device)
    for tensor_name, tensor in encoded_batch.items()
}

print("Model device:", next(model.parameters()).device)
print("Input device:", model_inputs["input_ids"].device)

Model device: cpu
Input device: cpu


In [27]:
# Finally Run inference on all the messages
# Understand logits
# Logits are the model's raw output scores.
# They:
# - Can be positive or negative
# - Do not need to be between zero and one
# - Do not add up to one
# - Are not probabilities
# This model produces five logits for each input because it has five possible sentiment labels.


with torch.no_grad():
    model_outputs = model(**model_inputs)

logits = model_outputs.logits

print("Logits:")
print(logits)

print("\nLogits shape:", logits.shape)

Logits:
tensor([[-1.5936, -1.5160, -1.1502,  1.6781,  1.6140],
        [ 0.6436,  2.3792, -0.2173, -1.7682, -1.2545],
        [-1.5804, -1.2787,  1.6774,  0.9049, -0.7937],
        [-1.2994,  3.2782, -0.5704,  0.7537, -1.4008],
        [ 2.7808, -0.8750, -1.0690, -1.2252, -0.9212],
        [ 1.3557,  2.0897, -0.9012, -1.5067, -1.4482],
        [-1.6044, -1.4256, -0.5985,  2.3975,  0.5048],
        [-1.4215, -1.5380, -1.0670,  2.4048,  0.8969],
        [-1.3514, -1.4558, -1.2242,  2.5681,  0.7096],
        [-1.4036, -1.4207, -1.1290,  2.2612,  1.0077],
        [ 1.3285,  2.1541, -1.0982, -1.3652, -1.3610],
        [ 1.5336,  1.9334, -1.3085, -1.3358, -1.3342],
        [ 2.1879,  0.6104, -1.2077, -1.3380, -1.2587]])

Logits shape: torch.Size([13, 5])


In [28]:
# Convert logits to probabilities with softmax

# Softmax converts each row of logits into a probability distribution.
# `dim=-1` means that softmax operates across the final dimension, which contains the five sentiment labels.
# Each resulting row should add up to approximately `1.0`.

probabilities = torch.softmax(
    logits,
    dim=-1
)

print("Probabilities:")
print(probabilities)

print("\nProbability totals:")
print(probabilities.sum(dim=-1))

Probabilities:
tensor([[0.0183, 0.0198, 0.0285, 0.4817, 0.4518],
        [0.1363, 0.7734, 0.0576, 0.0122, 0.0204],
        [0.0235, 0.0318, 0.6109, 0.2822, 0.0516],
        [0.0092, 0.8921, 0.0190, 0.0714, 0.0083],
        [0.9174, 0.0237, 0.0195, 0.0167, 0.0226],
        [0.3025, 0.6302, 0.0317, 0.0173, 0.0183],
        [0.0147, 0.0176, 0.0403, 0.8059, 0.1214],
        [0.0168, 0.0150, 0.0240, 0.7730, 0.1711],
        [0.0163, 0.0147, 0.0185, 0.8223, 0.1282],
        [0.0187, 0.0184, 0.0246, 0.7299, 0.2084],
        [0.2851, 0.6510, 0.0252, 0.0193, 0.0194],
        [0.3755, 0.5600, 0.0219, 0.0213, 0.0213],
        [0.7685, 0.1587, 0.0258, 0.0226, 0.0245]])

Probability totals:
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000])


In [29]:
# Select predictions with argmax

predicted_label_ids = torch.argmax(
    probabilities,
    dim=-1
)

print("Predicted label IDs:")
print(predicted_label_ids)


Predicted label IDs:
tensor([3, 1, 2, 1, 0, 1, 3, 3, 3, 3, 1, 1, 0])


In [30]:
# Create a readable results table

# Move results back to the CPU before converting them to Python values

probabilities_cpu = probabilities.detach().cpu()
predicted_label_ids_cpu = predicted_label_ids.detach().cpu().tolist()

classification_results = []

for message_index, message in enumerate(customer_messages):
    predicted_label_id = predicted_label_ids_cpu[message_index]
    predicted_label = config.id2label[predicted_label_id]

    confidence = probabilities_cpu[
        message_index,
        predicted_label_id
    ].item()

    classification_results.append({
        "message": message,
        "predicted_label_id": predicted_label_id,
        "sentiment": predicted_label,
        "confidence": confidence
    })

results_df = pd.DataFrame(classification_results)

results_df

,message,predicted_label_id,sentiment,confidence
0,The new version is fantastic. Everything works...,3,Positive,0.481687
1,My payment failed twice and support has not re...,1,Negative,0.773364
2,The transfer completed successfully.,2,Neutral,0.610932
3,"The app is okay, but the new interface takes s...",1,Negative,0.892084
4,This is the worst banking experience I have ev...,0,Very Negative,0.917426
5,The app constantly crashes and I cannot access...,1,Negative,0.630234
6,The customer service was excellent!,3,Positive,0.805933
7,El servicio al cliente fue excelente!,3,Positive,0.773020
8,Le service client était excellent !,3,Positive,0.822254
9,Der Kundenservice war ausgezeichnet!,3,Positive,0.729934


In [31]:
# Show every label probability
# A single predicted label hides potentially useful information. The full distribution shows which alternative labels the model considered.

detailed_results = []

for message_index, message in enumerate(customer_messages):
    result = {
        "message": message
    }

    for label_id, label_name in config.id2label.items():
        result[label_name] = probabilities_cpu[
            message_index,
            label_id
        ].item()

    detailed_results.append(result)

probability_df = pd.DataFrame(detailed_results)

probability_df

,message,Very Negative,Negative,Neutral,Positive,Very Positive
0,The new version is fantastic. Everything works...,0.018277,0.019751,0.028476,0.481687,0.451808
1,My payment failed twice and support has not re...,0.136338,0.773364,0.057644,0.012224,0.020430
2,The transfer completed successfully.,0.023505,0.031781,0.610932,0.282163,0.051619
3,"The app is okay, but the new interface takes s...",0.009170,0.892084,0.019010,0.071450,0.008286
4,This is the worst banking experience I have ev...,0.917426,0.023707,0.019527,0.016703,0.022637
5,The app constantly crashes and I cannot access...,0.302495,0.630234,0.031664,0.017283,0.018323
6,The customer service was excellent!,0.014733,0.017619,0.040286,0.805933,0.121429
7,El servicio al cliente fue excelente!,0.016845,0.014991,0.024011,0.773020,0.171133
8,Le service client était excellent !,0.016322,0.014704,0.018535,0.822254,0.128184
9,Der Kundenservice war ausgezeichnet!,0.018693,0.018376,0.024598,0.729934,0.208399


In [32]:
# Add a confidence threshold
# A classifier should not always be treated as correct.
# For this educational example:
# - Predictions at or above 70% confidence are accepted.
# - Predictions below 70% are sent for human review.

# This threshold is illustrative. A production threshold must be selected using validation data.
# Also, a softmax score is not guaranteed to be a well-calibrated real-world probability. A model can be confidently wrong.

CONFIDENCE_THRESHOLD = 0.70

results_df["decision"] = results_df["confidence"].apply(
    lambda confidence: (
        "Accept prediction"
        if confidence >= CONFIDENCE_THRESHOLD
        else "Needs human review"
    )
)

results_df

,message,predicted_label_id,sentiment,confidence,decision
0,The new version is fantastic. Everything works...,3,Positive,0.481687,Needs human review
1,My payment failed twice and support has not re...,1,Negative,0.773364,Accept prediction
2,The transfer completed successfully.,2,Neutral,0.610932,Needs human review
3,"The app is okay, but the new interface takes s...",1,Negative,0.892084,Accept prediction
4,This is the worst banking experience I have ev...,0,Very Negative,0.917426,Accept prediction
5,The app constantly crashes and I cannot access...,1,Negative,0.630234,Needs human review
6,The customer service was excellent!,3,Positive,0.805933,Accept prediction
7,El servicio al cliente fue excelente!,3,Positive,0.773020,Accept prediction
8,Le service client était excellent !,3,Positive,0.822254,Accept prediction
9,Der Kundenservice war ausgezeichnet!,3,Positive,0.729934,Accept prediction
